# Probar Jev, el modelo original de TypeSafe

[Jev](https://typesafe.ai/) es el modelo de **TypeSafe AI**, la empresa que lanzó la idea de los modelos "System One":
modelos que no escriben texto, sino que **toman decisiones** entre opciones y dan un porcentaje de seguridad.
CLM (el otro notebook) es una alternativa libre que copia esta misma forma de trabajar.

**Diferencia clave:** Jev es un servicio de pago al que se accede por internet (API), no se puede descargar.
Por eso este notebook **no necesita GPU**: funciona en Colab gratis o en cualquier ordenador.

**Qué necesitas:**
1. Una cuenta en [console.typesafe.ai](https://console.typesafe.ai/) (entrar con Google o con un código por correo).
2. Una clave de API, que se crea en [console.typesafe.ai/keys](https://console.typesafe.ai/keys).
3. Saldo en la cuenta. Al abrir el registro (20/09/2026) regalaban 5 $, pero según varias webs desde el 27/09/2026 las cuentas nuevas
   no reciben crédito gratis. Aun así es **muy barato**: 0,042 $ por millón de tokens de entrada (la salida es gratis).
   **Todo este notebook cuesta menos de 1 céntimo.**

En Colab, guarda la clave en **Secretos** (icono de la llave, a la izquierda) con el nombre `TYPESAFE_API_KEY`.

## 1. Instalación y clave

In [ ]:
!pip install -q typesafe-sdk datasets

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
except Exception:
    if not os.environ.get("TYPESAFE_API_KEY"):
        from getpass import getpass
        os.environ["TYPESAFE_API_KEY"] = getpass("Pega tu clave de TypeSafe: ")

from typesafe_sdk import TypeSafeClient, Choice, Noul, Score
client = TypeSafeClient()   # usa jev-latest por defecto

for m in client.models.list().models:
    print(m.name, m.release_date, "-", m.description)

In [ ]:
# Contador de gasto: cada llamada suma los tokens de entrada que cobra TypeSafe
import time
PRECIO_POR_TOKEN = 0.042 / 1_000_000
gasto = {"llamadas": 0, "tokens": 0}

def preguntar(state, questions):
    t = time.perf_counter()
    r = client.system_one(state=state, questions=questions)
    ms = (time.perf_counter() - t) * 1000
    gasto["llamadas"] += 1
    gasto["tokens"] += r.usage.input_tokens or 0
    return r, ms

def resumen_gasto():
    print(f"{gasto['llamadas']} llamadas | {gasto['tokens']:,} tokens | {gasto['tokens'] * PRECIO_POR_TOKEN:.5f} $")

## 2. Preguntas tipadas

Los mismos tres tipos que CLM:
- `Noul`: sí o no → probabilidad de que sí.
- `Choice`: elegir una opción.
- `Score`: puntuar en una escala.

Jev añade la **confianza** (`confidence`): lo seguro que está de su respuesta, para decidir si actuar solo o pasárselo a una persona.

In [ ]:
state = "Customer: my invoice was charged twice and nobody answers the phone!"
r, ms = preguntar(state, {
    "urgency": Noul(instructions="Is this urgent?"),
    "department": Choice(instructions="Which team should handle this?",
                         criteria={"billing": "Charges, invoices, refunds",
                                   "technical": "Bugs and outages",
                                   "sales": "New plans, pricing questions, upgrades"}),
    "frustration": Score(instructions="How frustrated is the customer?",
                         criteria=["Calm", "Frustrated", "Very angry"]),
})
print(f"modelo: {r.model} | {ms:.0f} ms | {r.usage.input_tokens} tokens")
print("urgente:", r.nouls["urgency"].noul)
print("departamento:", r.choices["department"].choice, r.choices["department"].probabilities,
      "confianza", r.choices["department"].confidence)
print("enfado (0-2):", r.scores["frustration"].score, "confianza", r.scores["frustration"].confidence)

In [ ]:
# El mismo caso en español (TypeSafe dice que funciona mejor en inglés)
r, ms = preguntar("Cliente: ¡me habéis cobrado la factura dos veces y nadie coge el teléfono!", {
    "department": Choice(instructions="¿Qué equipo debe encargarse?",
                         criteria={"billing": "Cobros, facturas, reembolsos",
                                   "technical": "Errores y caídas del servicio",
                                   "sales": "Planes nuevos, precios, mejoras de plan"}),
})
a = r.choices["department"]
print(a.choice, a.probabilities, "confianza", a.confidence, f"| {ms:.0f} ms")

## 3. Elegir herramienta y elegir la mejor solución

Las mismas pruebas que en el notebook de CLM.

In [ ]:
tools = {
    "get_weather": "Get the current weather forecast for a city",
    "send_email": "Send an email to a recipient with a subject and body",
    "search_flights": "Search flights between two airports on a date",
    "create_calendar_event": "Create a meeting in the user's calendar",
}
r, ms = preguntar("User: book me something to fly from Madrid to Paris next Friday",
                  {"tool": Choice(instructions="Which tool should the agent call next?", criteria=tools)})
print(r.choices["tool"].choice, r.choices["tool"].probabilities, f"| {ms:.0f} ms")

soluciones = {
    "iterativa": "def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a",
    "recursiva_sin_caso_base": "def fib(n):\n    return fib(n-1) + fib(n-2)",
    "formula_incorrecta": "def fib(n):\n    return n * (n + 1) // 2",
}
r, ms = preguntar("Write a Python function that returns the n-th Fibonacci number (fib(0)=0).",
                  {"best": Choice(instructions="Which candidate solution is correct?", criteria=soluciones)})
print(r.choices["best"].choice, r.choices["best"].probabilities, f"| {ms:.0f} ms")

## 4. Prueba de clasificación: AG News

Las **mismas 400 noticias** (misma semilla) y las mismas descripciones de categoría que en el notebook de CLM,
para poder comparar los resultados directamente.

In [ ]:
from datasets import load_dataset
import numpy as np

N = 400
ds = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=0).select(range(N))
labels = ["World", "Sports", "Business", "Sci/Tech"]
criteria = {
    "World": "World news: international politics, conflicts, governments, diplomacy",
    "Sports": "Sports news: games, teams, athletes, tournaments",
    "Business": "Business news: companies, markets, economy, finance",
    "Sci/Tech": "Science and technology news: computers, internet, software, space, research",
}
pregunta = {"topic": Choice(instructions="What is the topic of this news article?", criteria=criteria)}

preds, confs, lat = [], [], []
tokens_antes = gasto["tokens"]
for ex in ds:
    r, ms = preguntar(ex["text"], pregunta)
    a = r.choices["topic"]
    preds.append(labels.index(a.choice)); confs.append(a.confidence); lat.append(ms)

preds, confs, gold = np.array(preds), np.array(confs), np.array(ds["label"])
acierto = float((preds == gold).mean())
coste = (gasto["tokens"] - tokens_antes) * PRECIO_POR_TOKEN
print(f"Jev: accuracy={acierto:.3f} | latencia mediana={np.median(lat):.0f} ms | coste de las {N} noticias = {coste:.5f} $")

### ¿Su confianza sirve para algo?

Si la confianza está bien hecha, cuando Jev dice que está muy seguro debería acertar casi siempre.
Así se puede automatizar lo seguro y mandar a revisión solo lo dudoso.

In [ ]:
import pandas as pd
filas = []
for umbral in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]:
    sel = confs >= umbral
    filas.append({"confianza mínima": umbral,
                  "% de noticias que se automatizan": round(100 * sel.mean(), 1),
                  "acierto en esas noticias": round(float((preds[sel] == gold[sel]).mean()), 3) if sel.any() else None})
pd.DataFrame(filas)

## 5. Probar sus puntos débiles

La propia TypeSafe publica una lista de cosas que Jev hace mal
([Jev 1.13 jaggedness](https://docs.typesafe.ai/model-jaggedness/jev-1.13)): contar, hacer cuentas, comparar fechas,
preguntas con negaciones dobles… Lo comprobamos con unos casos sencillos cuya respuesta conocemos.

In [ ]:
casos = [
    # (tipo, estado, pregunta, opciones, respuesta correcta)
    ("contar", "apple, car, banana, house, orange, pear, table, grape",
     "How many fruits are in the list?", {"3": None, "4": None, "5": None, "6": None}, "5"),
    ("contar", "The word 'strawberry'",
     "How many times does the letter r appear in the word?", {"1": None, "2": None, "3": None, "4": None}, "3"),
    ("cuentas", "A shop sells pens at 3 euros each. Ana buys 7 pens and pays with a 50 euro note.",
     "How much change does Ana receive?", {"21": None, "29": None, "31": None, "47": None}, "29"),
    ("cuentas", "Order total: 1,250.40 USD. Discount: 15%.",
     "What is the total after the discount?", {"1062.84": None, "1100.00": None, "1235.40": None, "187.56": None}, "1062.84"),
    ("fechas", "Invoice A is dated 2026-03-09. Invoice B is dated 2026-09-03.",
     "Which invoice is older?", {"A": None, "B": None}, "A"),
    ("fechas", "The contract started on 15 January 2026 and lasts 90 days. Today is 20 April 2026.",
     "Is the contract still active today?", {"yes": None, "no": None}, "no"),
    ("negación", "The customer said: 'I don't not want the refund.'",
     "Does the customer want the refund?", {"yes": None, "no": None}, "yes"),
    ("sentido común", "Customer: the app crashes every time I open the settings page.",
     "Which team should handle this?", {"billing": None, "technical": None, "sales": None}, "technical"),
    ("sentido común", "Review: 'Absolutely loved it, would buy again!'",
     "What is the sentiment of the review?", {"positive": None, "negative": None, "neutral": None}, "positive"),
]

filas = []
for tipo, estado, instr, opciones, correcta in casos:
    r, _ = preguntar(estado, {"q": Choice(instructions=instr, criteria=opciones)})
    a = r.choices["q"]
    filas.append({"tipo": tipo, "pregunta": instr, "correcta": correcta, "Jev": a.choice,
                  "¿acierta?": "sí" if a.choice == correcta else "NO", "confianza": round(a.confidence, 2)})
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(filas)

## 6. Resumen y gasto

In [ ]:
resumen_gasto()

import json
with open("jev_resultados.json", "w") as f:
    json.dump({"modelo": r.model, "ag_news_accuracy": acierto, "ag_news_latencia_ms": float(np.median(lat)),
               "ag_news_predicciones": preds.tolist(), "ag_news_confianzas": confs.tolist(),
               "puntos_debiles": filas, "tokens": gasto["tokens"]}, f, indent=2, ensure_ascii=False)
print("Resultados guardados en jev_resultados.json")